# Clean observation data with pandas

The [Evidence Explorer](../README.md) combines a CSV of healthcare workflow observations with a separate CSV of source documents. Use the same fictional inputs as the app to see how to take a messy table all the way to a topic summary. The goal is to clean the data without losing where it came from.

After installing the repo dependencies, open this notebook with the same Python environment. For a browser notebook, install JupyterLab in that environment with `python -m pip install jupyterlab`, then run `python -m jupyterlab` from the repo root. Run the cells in order. The notebook also works when the kernel starts in the `notebooks` folder.

This walkthrough was prepared with AI assistance.

## 1. Inspect the input before changing it

Read both tables as strings so pandas does not infer numeric types for identifiers. Look at the shape and take a small sample of identifiers and labels. The `repr` view shows leading and trailing spaces that are easy to miss in the default table display.

In [1]:
from pathlib import Path

import pandas as pd

here = Path.cwd()
root = next(
    (path for path in (here, *here.parents) if (path / "data" / "evidence.csv").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Start the kernel from the repo or its notebooks folder.")

raw = pd.read_csv(root / "data" / "evidence.csv", dtype="string")
source_raw = pd.read_csv(root / "data" / "sources.csv", dtype="string")

print("Observation table shape:", raw.shape)
print("Source table shape:", source_raw.shape)
print(raw[["record_id", "source_id", "topic"]].tail(4).to_string(index=False))
print("Raw topic labels:", [repr(label) for label in raw["topic"].unique()])

Observation table shape: (9, 7)
Source table shape: (8, 5)
record_id source_id             topic
    ev-06       s06 Medication access
    ev-07       s07        scheduling
    ev-08       s08       Transitions
   EV-02       S02  medication access
Raw topic labels: ["'Referrals'", "' Medication Access '", "'Scheduling'", "'referrals'", "' Referrals '", "'Medication access'", "'scheduling'", "'Transitions'", "'medication access'"]


## 2. Normalize the fields you use to match records

Trim stray whitespace and treat empty fields as missing. Case-fold the ID and category columns so differences like `EV-02` versus `ev-02` do not create separate groups. Leave the casing of titles and observations alone. Keep the raw tables available so you can compare the before and after.

In [2]:
rows = raw.copy()
for column in rows.columns:
    rows[column] = rows[column].str.strip().replace("", pd.NA)

for column in ("record_id", "source_id", "topic", "status"):
    rows[column] = rows[column].str.casefold()

sources = source_raw.copy()
sources["source_id"] = sources["source_id"].str.strip().str.casefold()

if rows["record_id"].isna().any() or sources["source_id"].isna().any():
    raise ValueError("Record IDs and source table IDs are required.")

print(rows[["record_id", "source_id", "topic"]].tail(4).to_string(index=False))
print("Clean topic labels:", sorted(rows["topic"].dropna().unique()))

record_id source_id             topic
    ev-06       s06 medication access
    ev-07       s07        scheduling
    ev-08       s08       transitions
    ev-02       s02 medication access
Clean topic labels: ['medication access', 'referrals', 'scheduling', 'transitions']


## 3. Remove exact repeats and expose conflicts

After normalization, the two rows for `ev-02` agree on every field. Drop exact repeats first, then check for IDs that still occur more than once. Using `drop_duplicates(subset="record_id")` immediately would keep whichever row happens to come first, even when two rows disagree.

In [3]:
repeat_ids = rows["record_id"].duplicated(keep=False)
print("Rows with a repeated ID:")
print(rows.loc[repeat_ids, ["record_id", "title", "status"]].to_string(index=False))

unique_rows = rows.drop_duplicates()
conflicts = unique_rows.loc[unique_rows["record_id"].duplicated(keep=False)]
if not conflicts.empty:
    bad_ids = conflicts["record_id"].unique().tolist()
    raise ValueError(f"Conflicting observations for {bad_ids}")

print(f"Removed {len(rows) - len(unique_rows)} exact repeat. Kept {len(unique_rows)} observations.")

Rows with a repeated ID:
record_id                                   title   status
    ev-02 Pharmacy transfer resets the call queue observed
    ev-02 Pharmacy transfer resets the call queue observed
Removed 1 exact repeat. Kept 8 observations.


## 4. Attach source details with a validated left join

An observation points to one source. A source may support several observations, so the relationship from observations to sources is **many to one**. `validate="many_to_one"` raises an error if the source table contains duplicate join keys. Without that check, a duplicate source row could quietly multiply observations.

A left join retains every observation. The merge indicator tells you whether each result matched a source record.

In [4]:
evidence = unique_rows.merge(
    sources,
    on="source_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
assert len(evidence) == len(unique_rows)

print(evidence["_merge"].value_counts().to_string())
print(evidence[["record_id", "source_ref", "publisher"]].head(3).to_string(index=False))

_merge
both          8
left_only     0
right_only    0
record_id  source_ref                    publisher
    ev-01 DEMO-OPS-01 Forest County Health Network
    ev-02  DEMO-WF-02    Lake Clinic Collaborative
    ev-03 DEMO-SUR-03        West Shore Access Lab


## 5. See the difference between a missing date and a missing source

`ev-06` has a matching interview note with an empty date. Keep that date missing after conversion. A matched source with a missing date still has provenance. Now simulate a broken join by dropping the source for `ev-06` from a temporary copy and see how the two cases look.

In [5]:
evidence["published"] = pd.to_datetime(
    evidence["published"], format="%Y-%m-%d", errors="coerce"
)
cols = ["record_id", "source_id", "source_ref", "published", "_merge"]

print("Real example with no date:")
print(evidence.loc[evidence["published"].isna(), cols].to_string(index=False))

simulated = unique_rows.merge(
    sources.loc[sources["source_id"].ne("s06")],
    on="source_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("\nSimulated example with no matching source:")
print(simulated.loc[simulated["_merge"].eq("left_only"), cols].to_string(index=False))

Real example with no date:
record_id source_id  source_ref published _merge
    ev-06       s06 DEMO-INT-06       NaT   both

Simulated example with no matching source:
record_id source_id source_ref published    _merge
    ev-06       s06       <NA>      <NA> left_only


## 6. Summarize topics without counting the same record twice

Count the cleaned observation rows in each topic. Use `size` so that a blank date does not remove a record from the count. Summarize date coverage separately. `dropna=False` would retain a group for any unlabeled topic.

In [6]:
summary = (
    evidence.assign(has_date=evidence["published"].notna())
    .groupby("topic", dropna=False)
    .agg(observations=("record_id", "size"), dated=("has_date", "sum"))
    .sort_values("observations", ascending=False)
)
summary["undated"] = summary["observations"] - summary["dated"]

assert summary["observations"].sum() == len(unique_rows)
print(summary.to_string())

                   observations  dated  undated
topic                                          
referrals                     3      3        0
medication access             2      1        1
scheduling                    2      2        0
transitions                   1      1        0


## 7. Compare your result with the app

The app uses [`load_evidence`](../evidence.py), which packages the same transformations and formats dates as strings for JSON. Check your results against the implementation used by the browser. This compares the entire data frame so a matching row count cannot hide wrong source details.

In [7]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from evidence import load_evidence

app_evidence, audit = load_evidence()
walkthrough_evidence = evidence.drop(columns="_merge").sort_values("record_id").reset_index(drop=True)
walkthrough_evidence["published"] = walkthrough_evidence["published"].dt.strftime("%Y-%m-%d")
pd.testing.assert_frame_equal(walkthrough_evidence, app_evidence)
print(f"Matches the app: {audit['input_rows']} inputs, {audit['duplicates']} repeat, {len(app_evidence)} observations.")

Matches the app: 9 inputs, 1 repeat, 8 observations.


## Try it yourself

1. Filter to medication access observations and show each title, source reference, and date. Which source would you follow up with to fill in the date?
2. Make a temporary copy of the sources with one extra copy of the first row. Re-run the validated merge. Read the error and explain which part of the data would otherwise be inflated.

<details>
<summary>Check your work</summary>

For the first question, use `evidence.loc[evidence["topic"].eq("medication access"), ["title", "source_ref", "published"]]`. Two observations match, and the date is absent for the interview note `DEMO-INT-06` from Lake Clinic Collaborative.

For the second, try the following cell fragment. It should raise `pandas.errors.MergeError`. A source duplicated in the right table would make matching observations appear twice and inflate their topic counts.

```python
bad_sources = pd.concat([sources, sources.iloc[[0]]], ignore_index=True)
unique_rows.merge(bad_sources, on="source_id", how="left", validate="many_to_one")
```

</details>